# 📸 Dataset Zuzi — SFW (bikini max)

Generuje zestaw zdjęć postaci **Zuzia** do treningu LoRA. Wyłącznie treści SFW:
strój kąpielowy to maksimum, zero nagości, zero bielizny, zero prześwitów.

**Ustawienia:** 6 stylizacji × 3 ujęcia = 18 zdjęć (partia pilotażowa).
Po akceptacji jakości zwiększ `N_PER_OUTFIT` w komórce 1.

**Wymaga:** LoRA `zuzia_v3_1400.safetensors` na Dysku (root lub `AI-Colab/models/loras`).


In [ ]:
#@title 1. Konfiguracja { display-mode: "form" }
N_PER_OUTFIT = 3 #@param {type:"integer"}
LORA_WEIGHT = 0.85 #@param {type:"slider", min:0, max:1.5, step:0.05}
STEPS = 30 #@param {type:"integer"}
WIDTH, HEIGHT = 832, 1216
OUT_DIR = "/content/drive/MyDrive/AI-Colab/dataset_zuzia_sfw"

OUTFITS = [
    ("casual",  "white cotton t-shirt, blue jeans, white sneakers"),
    ("summer",  "light floral summer dress, sandals"),
    ("sport",   "black athletic leggings, grey sports bra, running shoes"),
    ("bikini",  "beige two-piece swimsuit, beach"),
    ("street",  "oversized beige hoodie, denim shorts, white sneakers"),
    ("elegant", "elegant black midi dress, small handbag"),
]
POSES = [
    "standing naturally, hands relaxed, looking at camera",
    "walking towards camera, candid moment",
    "sitting on stairs, relaxed posture, soft smile",
]

NEGATIVE = ("nude, naked, topless, nipples, areola, lingerie, underwear, thong, see-through, "
            "transparent clothing, exposed breasts, exposed genitals, sexual pose, child, teen, "
            "cartoon, anime, 3d render, cgi, plastic skin, airbrushed, deformed, extra limbs, "
            "extra fingers, bad hands, watermark, text, logo")


In [ ]:
#@title 2. LoRA + model { display-mode: "form" }
import glob, os, subprocess, sys

!pip -q install -U "diffusers>=0.31" "transformers>=4.44" accelerate safetensors

from google.colab import drive
drive.mount("/content/drive")

CANDIDATES = [
    "/content/drive/MyDrive/AI-Colab/models/loras/zuzia_v3_1400.safetensors",
    "/content/drive/MyDrive/zuzia_v3_1400.safetensors",
]
CANDIDATES += sorted(glob.glob("/content/drive/MyDrive/**/zuzia_v3_1400.safetensors", recursive=True))
LORA = next((p for p in CANDIDATES if os.path.exists(p)), None)
print("LoRA:", LORA)
if not LORA:
    raise SystemExit("Brak LoRA na Dysku — wgraj zuzia_v3_1400.safetensors do My Drive.")

import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler

for variant in ("fp16", None):
    try:
        pipe = DiffusionPipeline.from_pretrained(
            "SG161222/RealVisXL_V5.0", torch_dtype=torch.float16,
            variant=variant, use_safetensors=True)
        break
    except Exception as e:
        print("retry without variant:", type(e).__name__)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe.load_lora_weights(LORA, adapter_name="zuzia")
pipe.set_adapters(["zuzia"], adapter_weights=[LORA_WEIGHT])
pipe.to("cuda")
pipe.enable_vae_tiling()
print("Model gotowy:", pipe.__class__.__name__)


In [ ]:
#@title 3. Generuj { display-mode: "form" }
import os, time, zipfile
from PIL import Image

os.makedirs(OUT_DIR, exist_ok=True)
log = []
t0 = time.time()

for oi, (slug, outfit) in enumerate(OUTFITS):
    for pi in range(N_PER_OUTFIT):
        pose = POSES[pi % len(POSES)]
        seed = 42000 + oi * 100 + pi
        prompt = (f"RAW photo, portrait of zuzia, a 25 year old woman with long dark hair, "
                  f"curvy hourglass figure, natural body proportions, wearing {outfit}, "
                  f"{pose}, outdoor daylight, natural skin texture, film grain, 85mm lens, "
                  f"sharp focus, high detail")
        g = torch.Generator(device="cuda").manual_seed(seed)
        img = pipe(prompt=prompt, negative_prompt=NEGATIVE, width=WIDTH, height=HEIGHT,
                   num_inference_steps=STEPS, guidance_scale=5.5, generator=g).images[0]
        name = f"zuzia_{slug}_{pi+1:02d}.png"
        img.save(os.path.join(OUT_DIR, name))
        log.append({"file": name, "seed": seed, "outfit": slug, "prompt": prompt})
        print(f"[{len(log):02d}] {name}  ({time.time()-t0:.0f}s)")

import json
json.dump(log, open(os.path.join(OUT_DIR, "metadata.json"), "w"), ensure_ascii=False, indent=1)

zpath = OUT_DIR + ".zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT_DIR):
        z.write(os.path.join(OUT_DIR, f), f)
print(f"\nGotowe: {len(log)} zdjęć w {time.time()-t0:.0f}s")
print("Folder:", OUT_DIR)
print("ZIP:", zpath)
